# Base CNN — Behavior Cloning

Spatial CNN policy over the board (backbone + 1×1 policy head).

- Input `(14, 23, 23)` → backbone `(32, 23, 23)` → policy `(9, 23, 23)`
- Channels `0–7`: move `dir + 4*split` from cell `(r, c)`; channel `8`: pass
- Reproducible `game_id` split · epoch loss/acc · save `bc/checkpoints/base_cnn.pt`

## 1. Imports & config

**Colab data flow**
1. Mount Drive (`MyDrive/general_data`)
2. Copy **all** `bc_shard_*.npz` → `/content/bc/data` (once per runtime)
3. Index all shards once · re-run **split only** to try different train/val/test fracs


In [1]:
# Colab only — mount Drive. Data lives at MyDrive/general_data.
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass

DRIVE_DATA = "/content/drive/MyDrive/general_data"
LOCAL_DATA = "/content/bc/data"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Copy ALL shards Drive → Colab local SSD (once per runtime).
# Re-run split / train freely after this; skip re-copy if local already complete.
import shutil
from pathlib import Path

DRIVE_DATA = Path(globals().get("DRIVE_DATA") or "/content/drive/MyDrive/general_data")
LOCAL_DATA = Path("/content/bc/data")

drive_shards = sorted(DRIVE_DATA.glob("bc_shard_*.npz")) if DRIVE_DATA.exists() else []
local_shards = sorted(LOCAL_DATA.glob("bc_shard_*.npz")) if LOCAL_DATA.exists() else []

print(f"drive: {DRIVE_DATA}  shards={len(drive_shards)}")
print(f"local: {LOCAL_DATA}  shards={len(local_shards)}")

if local_shards and (not drive_shards or len(local_shards) >= len(drive_shards)):
    print(f"skip copy — local already has {len(local_shards)} shard(s) (full dataset)")
elif not drive_shards:
    print("no Drive shards found — mount Drive / check DRIVE_DATA")
else:
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    print(f"copying ALL {len(drive_shards)} shards → {LOCAL_DATA} ...")
    for i, src in enumerate(drive_shards, 1):
        dst = LOCAL_DATA / src.name
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            status = "skip"
        else:
            shutil.copy2(src, dst)
            status = "copied"
        if i == 1 or i == len(drive_shards) or i % 10 == 0 or status == "copied":
            print(f"  [{i}/{len(drive_shards)}] {status}  {src.name}")
    local_shards = sorted(LOCAL_DATA.glob("bc_shard_*.npz"))
    assert len(local_shards) >= len(drive_shards), (
        f"incomplete copy: local={len(local_shards)} drive={len(drive_shards)}"
    )
    print(f"done. local has ALL {len(local_shards)} shards — safe to change splits without re-copy")


drive: /content/drive/MyDrive/general_data  shards=74
local: /content/bc/data  shards=74
skip copy — local already has 74 shard(s) (full dataset)


In [3]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

_ROOT = Path.cwd().resolve()
if not (_ROOT / "generals").exists() and (_ROOT.parent / "generals").exists():
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

LOCAL_DATA = Path(globals().get("LOCAL_DATA") or "/content/bc/data")
DRIVE_DATA = Path(globals().get("DRIVE_DATA") or "/content/drive/MyDrive/general_data")

# Prefer local SSD copy of ALL shards (from copy cell). Drive is fallback only.
DATA_DIRS = [
    LOCAL_DATA,
    DRIVE_DATA,
    Path("bc/data"),
    Path("data"),
    Path("bc/bc/data"),
]
DATA_DIRS = [d for d in DATA_DIRS if d is not None]
DATA_DIR = next(
    (d for d in DATA_DIRS if d.exists() and list(d.glob("bc_shard_*.npz"))),
    DATA_DIRS[0],
)

GRID_SIZE = 23
# Training / paper-aligned hyperparams
NENVS = 128
NSTEPS = 6000
BATCH_SIZE = 600
TRUNCATION = 2000
LR = 5e-5          # heads training
LR_FINETUNE = 2e-5  # fine-tuning (unused unless you switch)
GAMMA = 1.0
GRAD_CLIP_NORM = 0.25
MAX_EPOCHS = 20
PATIENCE = 5
SEED = 0

TRAIN_FRAC = 0.4
VAL_FRAC = 0.10
TEST_FRAC = 0.10

CKPT_DIR = Path("/content/drive/MyDrive/generals_ai/bc/checkpoints")
CKPT_NAME = "base_cnn.pt"

DEVICE = torch.device("cpu")
n_shards = len(list(DATA_DIR.glob("bc_shard_*.npz"))) if DATA_DIR.exists() else 0
print(f"device: {DEVICE}")
print(f"data dir: {DATA_DIR.resolve()}  exists={DATA_DIR.exists()}  shards={n_shards}")
print(f"using LOCAL SSD" if DATA_DIR.resolve() == LOCAL_DATA.resolve() else "WARNING: reading from Drive (slow) — run copy cell")
print(f"split: train={TRAIN_FRAC:.0%} val={VAL_FRAC:.0%} test={TEST_FRAC:.0%}  seed={SEED}")
print(f"train: batch={BATCH_SIZE} lr={LR} clip={GRAD_CLIP_NORM} epochs<={MAX_EPOCHS}  nenvs={NENVS} nsteps={NSTEPS} trunc={TRUNCATION} γ={GAMMA}")


device: cpu
data dir: /content/bc/data  exists=True  shards=74
using LOCAL SSD
split: train=40% val=10% test=10%  seed=0
train: batch=600 lr=5e-05 clip=0.25 epochs<=20  nenvs=128 nsteps=6000 trunc=2000 γ=1.0


## 2. Index all shards (lazy — do not load full `obs` into RAM)

Indexes **every** local shard once (`game_id` only). Re-run §3 to change splits without re-copy or re-index.


In [4]:
REQUIRED_KEYS = ("obs", "actions", "game_id", "player", "turn", "aug")

shard_paths_all = sorted(DATA_DIR.glob("bc_shard_*.npz"))
assert shard_paths_all, f"no shards found under {DATA_DIR}"

shard_paths: list[Path] = []
skipped: list[str] = []

for path in shard_paths_all:
    try:
        with np.load(path, allow_pickle=False) as z:
            keys = set(z.files)
            if not set(REQUIRED_KEYS).issubset(keys):
                skipped.append(f"{path.name} (missing {sorted(set(REQUIRED_KEYS) - keys)})")
                continue
            _ = z["game_id"][:1]
        shard_paths.append(path)
    except Exception as e:
        skipped.append(f"{path.name} ({type(e).__name__}: {e})")

if skipped:
    print(f"skipped {len(skipped)} incomplete/corrupt shard(s):")
    for s in skipped:
        print(f"  - {s}")

assert shard_paths, "no valid shards left after filtering"

shard_game_ids: list[np.ndarray] = []
all_game_ids_list: list[np.ndarray] = []
sample_index: list[tuple[int, int]] = []

for si, path in enumerate(shard_paths):
    with np.load(path, allow_pickle=False) as z:
        gids = z["game_id"].astype(str)
    shard_game_ids.append(gids)
    for li in range(len(gids)):
        sample_index.append((si, li))
    all_game_ids_list.append(gids)

game_ids = np.concatenate(all_game_ids_list, axis=0)
unique_games = np.unique(game_ids)
print(f"shards: {len(shard_paths)} valid / {len(shard_paths_all)} total")
print(f"samples: {len(sample_index):,}")
print(f"unique games: {len(unique_games):,}")
print(f"first shard: {shard_paths[0].name}")
print(f"last shard:  {shard_paths[-1].name}")

shards: 74 valid / 74 total
samples: 9,655,528
unique games: 2,368
first shard: bc_shard_0000_n32_t1001.npz
last shard:  bc_shard_0073_n32_t1001.npz


## 3. Reproducible split by `game_id`

Change `TRAIN_FRAC` / `VAL_FRAC` / `TEST_FRAC` in config and **re-run only this section** — no need to re-copy or re-index.
Same `SEED` + same shards → same games for a given frac.


In [5]:
def split_by_game_id(
    game_ids: np.ndarray,
    train_frac: float,
    val_frac: float,
    test_frac: float,
    seed: int = SEED,
):
    """Deterministic game-level split. No sample leakage across splits."""
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-9 or (
        train_frac + val_frac + test_frac <= 1.0
    ), "fractions should sum to <= 1"

    unique = np.unique(game_ids)
    unique = np.sort(unique)
    rng = np.random.default_rng(seed)
    unique = rng.permutation(unique)

    n = len(unique)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    n_test = int(n * test_frac)

    train_ids = set(unique[:n_train].tolist())
    val_ids = set(unique[n_train : n_train + n_val].tolist())
    test_ids = set(unique[n_train + n_val : n_train + n_val + n_test].tolist())

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    train_mask = np.isin(game_ids, list(train_ids))
    val_mask = np.isin(game_ids, list(val_ids))
    test_mask = np.isin(game_ids, list(test_ids))

    print(
        f"games:   train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}  "
        f"unused={n - len(train_ids) - len(val_ids) - len(test_ids)}  seed={seed}"
    )
    print(
        f"samples: train={int(train_mask.sum()):,}  val={int(val_mask.sum()):,}  "
        f"test={int(test_mask.sum()):,}"
    )
    return train_mask, val_mask, test_mask, {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids,
        "seed": seed,
        "ordered_unique": unique,
    }


train_mask, val_mask, test_mask, split_ids = split_by_game_id(
    game_ids, TRAIN_FRAC, VAL_FRAC, TEST_FRAC, seed=SEED
)
assert split_ids["train"].isdisjoint(split_ids["val"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["val"].isdisjoint(split_ids["test"])
print("leakage check passed")

games:   train=947  val=236  test=236  unused=949  seed=0
samples: train=3,842,088  val=947,168  test=921,120
leakage check passed


## 4. Lazy multi-shard dataset + spatial CNN policy

In [6]:
N_POLICY = 9  # 4 dirs × 2 splits + pass
PASS_CH = 8


def action_to_flat_index(act, grid_size: int = GRID_SIZE) -> int:
    """Map [pass, row, col, dir, split] → class in flattened (9, H, W)."""
    to_pass, row, col, direction, split = (int(x) for x in act)
    if to_pass:
        return PASS_CH  # channel 8 at (0, 0)
    ch = int(direction) + 4 * int(split)
    return ch + N_POLICY * (int(col) + grid_size * int(row))


def flat_index_to_action(idx: int, grid_size: int = GRID_SIZE) -> tuple[int, int, int, int, int]:
    """Inverse → (pass, row, col, dir, split)."""
    ch = idx % N_POLICY
    cell = idx // N_POLICY
    row, col = divmod(cell, grid_size)
    if ch == PASS_CH:
        return (1, 0, 0, 0, 0)
    direction, split = ch % 4, ch // 4
    return (0, row, col, direction, split)


class ShardCache:
    """Keep one decompressed shard in RAM.

    Shards are savez_compressed — mmap is a no-op, so z['obs'] fully decompresses
    (~4GB/shard). Never globally shuffle across shards or training will thrash.
    """

    def __init__(self, paths: list[Path]):
        self.paths = paths
        self._idx = -1
        self._obs = None
        self._actions = None

    def get(self, shard_idx: int):
        if shard_idx != self._idx:
            path = self.paths[shard_idx]
            print(f"  [shard] load {path.name} ...", flush=True)
            with np.load(path, allow_pickle=False) as z:
                self._obs = np.asarray(z["obs"])
                self._actions = np.asarray(z["actions"])
            self._idx = shard_idx
            print(
                f"  [shard] ready {path.name}  obs={self._obs.shape}  "
                f"~{self._obs.nbytes / 1e9:.2f}GB",
                flush=True,
            )
        return self._obs, self._actions


class ShardGroupedSampler:
    """Epoch order: shuffle shards, then shuffle samples within each shard."""

    def __init__(self, entries: list[tuple[int, int]], seed: int = 0, shuffle: bool = True):
        self.entries = entries
        self.seed = seed
        self.shuffle = shuffle
        self.epoch = 0
        self._by_shard: dict[int, list[int]] = {}
        for ds_idx, (si, _) in enumerate(entries):
            self._by_shard.setdefault(si, []).append(ds_idx)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        shards = list(self._by_shard.keys())
        if self.shuffle:
            rng.shuffle(shards)
        for si in shards:
            idxs = self._by_shard[si].copy()
            if self.shuffle:
                rng.shuffle(idxs)
            yield from idxs

    def __len__(self):
        return len(self.entries)


class MultiShardBCDataset(Dataset):
    def __init__(self, shard_paths: list[Path], sample_index: list[tuple[int, int]], mask: np.ndarray):
        self.paths = shard_paths
        self.cache = ShardCache(shard_paths)
        idxs = np.nonzero(mask)[0]
        self.entries = [sample_index[i] for i in idxs]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i: int):
        si, li = self.entries[i]
        obs_arr, act_arr = self.cache.get(si)
        obs = torch.from_numpy(np.asarray(obs_arr[li], dtype=np.float32))
        act = np.asarray(act_arr[li], dtype=np.int64)
        y = torch.tensor(action_to_flat_index(act), dtype=torch.int64)
        return obs, y


class GeneralsBCNet(nn.Module):
    """Spatial CNN backbone + 1×1 policy head → (B, 9, H, W)."""

    def __init__(self, in_channels: int = 14):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.policy = nn.Conv2d(in_channels=32, out_channels=N_POLICY, kernel_size=1)

    def forward(self, x):
        return self.policy(self.backbone(x))


def policy_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    b = logits.size(0)
    return nn.functional.cross_entropy(logits.view(b, -1), y)


## 5. Train / eval helpers

In [ ]:
def set_seed(seed: int = SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    n = 0
    correct = 0

    for xb, y in loader:
        xb = xb.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(xb)
        loss = policy_loss(logits, y)
        total_loss += float(loss.item()) * xb.size(0)
        n += xb.size(0)
        pred = logits.view(xb.size(0), -1).argmax(dim=-1)
        correct += int((pred == y).sum())

    return {
        "loss": total_loss / max(n, 1),
        "acc": correct / max(n, 1),
    }


def fmt_metrics(prefix: str, m: dict[str, float]) -> str:
    return f"{prefix}_loss={m['loss']:.4f}  {prefix}_acc={m['acc']:.3f}"

: 

## 6. Train single model + save

In [8]:
set_seed(SEED)

train_ds = MultiShardBCDataset(shard_paths, sample_index, train_mask)
val_ds = MultiShardBCDataset(shard_paths, sample_index, val_mask)
test_ds = MultiShardBCDataset(shard_paths, sample_index, test_mask)

# Shard-grouped sampling: decompress each compressed .npz once per pass, not per batch.
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=ShardGroupedSampler(train_ds.entries, seed=SEED, shuffle=True),
    num_workers=0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    sampler=ShardGroupedSampler(val_ds.entries, seed=SEED, shuffle=False),
    num_workers=0,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    sampler=ShardGroupedSampler(test_ds.entries, seed=SEED, shuffle=False),
    num_workers=0,
)

model = GeneralsBCNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")
best_state = None
stale = 0
history = []

print(
    f"training base_cnn  |  train={len(train_ds):,}  val={len(val_ds):,}  "
    f"test={len(test_ds):,}  epochs<={MAX_EPOCHS}  patience={PATIENCE}"
)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n = 0
    correct = 0
    n_batches = 0

    for xb, y in train_loader:
        n_batches += 1
        if n_batches == 1 or n_batches % 50 == 0:
            print(f"  epoch {epoch:02d}  batch {n_batches}  samples={n + xb.size(0):,}", flush=True)
        xb = xb.to(DEVICE)
        y = y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = policy_loss(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        opt.step()

        running_loss += float(loss.item()) * xb.size(0)
        n += xb.size(0)
        with torch.no_grad():
            pred = logits.view(xb.size(0), -1).argmax(dim=-1)
            correct += int((pred == y).sum())

    train_metrics = {
        "loss": running_loss / max(n, 1),
        "acc": correct / max(n, 1),
    }
    val_metrics = evaluate(model, val_loader)

    history.append(
        {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
    )
    print(
        f"epoch {epoch:02d}  {fmt_metrics('train', train_metrics)}  |  "
        f"{fmt_metrics('val', val_metrics)}"
    )

    if val_metrics["loss"] < best_val - 1e-4:
        best_val = val_metrics["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f"early stopping at epoch {epoch} (best val_loss={best_val:.4f})")
            break

if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate(model, test_loader)
print(f"\nTEST  {fmt_metrics('test', test_metrics)}")
print(f"best val_loss={best_val:.4f}")

CKPT_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = CKPT_DIR / CKPT_NAME
torch.save(
    {
        "model": best_state if best_state is not None else model.state_dict(),
        "test": test_metrics,
        "best_val_loss": best_val,
        "history": history,
        "config": {
            "seed": SEED,
            "train_frac": TRAIN_FRAC,
            "val_frac": VAL_FRAC,
            "test_frac": TEST_FRAC,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "lr_finetune": LR_FINETUNE,
            "grad_clip_norm": GRAD_CLIP_NORM,
            "nenvs": NENVS,
            "nsteps": NSTEPS,
            "truncation": TRUNCATION,
            "gamma": GAMMA,
            "grid_size": GRID_SIZE,
            "n_policy": N_POLICY,
            "n_train_games": len(split_ids["train"]),
            "n_val_games": len(split_ids["val"]),
            "n_test_games": len(split_ids["test"]),
        },
        "split_game_ids": {
            "train": sorted(split_ids["train"]),
            "val": sorted(split_ids["val"]),
            "test": sorted(split_ids["test"]),
        },
    },
    ckpt_path,
)
print(f"saved {ckpt_path.resolve()}")


training base_cnn  |  train=3,842,088  val=947,168  test=921,120  epochs<=20  patience=5


: 

: 